In [ ]:
import lightkurve as lk
import numpy as np
import matplotlib.pyplot as plt

# STEP 1
target = "TRAPPIST-1"
tpf = lk.search_targetpixelfile(target).download()

# STEP 2
lc = tpf.to_lightcurve(aperture_mask=tpf.pipeline_mask)
lc = lc.remove_nans()

# STEP 3 (smaller window)
flat_lc = lc.flatten(window_length=201)

# STEP 4 (short periods)
periods = np.linspace(0.5, 5, 10000)
bls = flat_lc.to_periodogram(method="bls", period=periods)

best_period = bls.period_at_max_power

# STEP 5
folded_lc = flat_lc.fold(period=best_period, epoch_time=best_period/2)

# STEP 6
flux = folded_lc.flux
flux = flux[~np.isnan(flux)]
transit_depth = 1 - np.median(np.sort(flux)[:100])

rp_rs = np.sqrt(transit_depth)

# STEP 7
ax = folded_lc.plot()
ax.set_xlim(-0.05, 0.05)

plt.title("TRAPPIST-1 Transit")
plt.show()